# Homework 10 - ST 554

Author: Max Campbell

## Part 1 - Creating Streaming Data Using `rate`

In this assignment, we are tasked with creating data streaming processes to allow us to read in data over time. This is useful in big data contexts where we often need to read in large amounts of data at a high velocity. Luckily for us, Spark allows us to stream data! We can create testing data using the `rate` format to demonstrate the basics of this process. First, let's initialize the Spark session and the corresponding stream reader:

In [1]:
#Load necessary modules
from pyspark.sql import SparkSession
from pyspark.sql.functions import sqrt, try_mod, col, lit


#Initialize SparkSession
spark = SparkSession.builder.master('local[*]').appName('HW9') \
    .config("spark.sql.ansi.enabled", "false").getOrCreate()

#Set up the stream
df = spark \
    .readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 14:42:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


As with any dataset, we want to be able to perform operations on the observations we collect. We will demonstrate how to do this with Spark by finding the square root and modulus 4 of the value of interest in our `rate` data.

In [2]:
#Set up computations that we want to perform on the stream
#Note: lit function casts 4 to a Column object.
query = df.withColumn('sqrt', sqrt(col('value'))).withColumn('mod_4', try_mod(col('value'), lit(4)))

The last thing to do is start the stream! We'll let it run for about 30 seconds, having it write to the dataframe in-memory, and then (manually) stop it.

In [5]:
#Start stream, having it write to memory
current_stream = query.writeStream.format("memory").queryName("rateTable").start()

26/04/17 14:42:28 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-3fae7eec-25f0-4320-b316-d3e4b3e11e58. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/17 14:42:28 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [6]:
#Stop output
current_stream.stop()

#Stop stream, and view the output
spark.sql("select * from rateTable").show()

26/04/17 14:43:13 WARN DAGScheduler: Failed to cancel job group fff03e17-67c8-4bad-8ff0-20f93bf08875. Cannot find active jobs for it.
26/04/17 14:43:13 WARN DAGScheduler: Failed to cancel job group fff03e17-67c8-4bad-8ff0-20f93bf08875. Cannot find active jobs for it.


+--------------------+-----+------------------+-----+
|           timestamp|value|              sqrt|mod_4|
+--------------------+-----+------------------+-----+
|2026-04-17 14:42:...|    0|               0.0|    0|
|2026-04-17 14:42:...|    1|               1.0|    1|
|2026-04-17 14:42:...|    2|1.4142135623730951|    2|
|2026-04-17 14:42:...|    3|1.7320508075688772|    3|
|2026-04-17 14:42:...|    4|               2.0|    0|
|2026-04-17 14:42:...|    5|  2.23606797749979|    1|
|2026-04-17 14:42:...|    6| 2.449489742783178|    2|
|2026-04-17 14:42:...|    7|2.6457513110645907|    3|
|2026-04-17 14:42:...|    8|2.8284271247461903|    0|
|2026-04-17 14:42:...|    9|               3.0|    1|
|2026-04-17 14:42:...|   10|3.1622776601683795|    2|
|2026-04-17 14:42:...|   11|   3.3166247903554|    3|
|2026-04-17 14:42:...|   12|3.4641016151377544|    0|
|2026-04-17 14:42:...|   13| 3.605551275463989|    1|
|2026-04-17 14:42:...|   14|3.7416573867739413|    2|
|2026-04-17 14:42:...|   15|

We've successfully streamed some data, found the square root and the mod 4 for the `value` column, and saved it in-memory!

## Part 2 - Using data from a CSV with a Pipeline

Now that we've demonstrated the basics, let's show a use case that goes beyond using basic `rate` data. We'll use some `bikeDetails` data as an example of how streaming might work in practice. In short, we will set up our stream to read in data from a specified file path, and append data to an existing dataset. We will also (eventually) want to fit this data to a model. As such, we will build a MLlib pipeline to use as our query in this context.

In [14]:
#Import necessary modules
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler

#Use existing spark session to read the "for fit" data as an SQL dataframe
bike = spark.read.csv("bikedata_staging/bikeDetails_for_fit.csv", header = True, inferSchema = True)
bike.show(10) #Verify output

#Set up transformers
sqlTrans = SQLTransformer(statement = "SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven, CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner FROM __THIS__")

vecAssembler = VectorAssembler(handleInvalid = "keep", outputCol = "features").setInputCols(["year", "log_km_driven", "one_owner"])

pipeline = Pipeline(stages = [sqlTrans, vecAssembler])

fitted_bike = pipeline.fit(bike)

+--------------------+-------------+----+-----------+---------+---------+-----------------+
|                name|selling_price|year|seller_type|    owner|km_driven|ex_showroom_price|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
|Royal Enfield Cla...|       175000|2019| Individual|1st owner|      350|             NULL|
|           Honda Dio|        45000|2017| Individual|1st owner|     5650|             NULL|
|Royal Enfield Cla...|       150000|2018| Individual|1st owner|    12000|           148114|
|Yamaha Fazer FI V...|        65000|2015| Individual|1st owner|    23000|            89643|
|Yamaha SZ [2013-2...|        20000|2011| Individual|2nd owner|    21000|             NULL|
|    Honda CB Twister|        18000|2010| Individual|1st owner|    60000|            53857|
|Honda CB Hornet 160R|        78500|2018| Individual|1st owner|    17000|            87719|
|Royal Enfield Bul...|       180000|2008| Individual|2nd owner|    39000|       

Now that we've created a pipeline to fit new data to, we can start building our stream.

In [20]:
#Set up stream reader
bike_read = spark \
    .readStream \
    .schema(bike.schema) \
    .option("header", "true") \
    .csv("bikedata")
    

#Set up pipeline to transform newly input data to the correct format
bike_transform = fitted_bike.transform(bike_read)

query = bike_transform.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

26/04/17 15:14:06 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-8a18cf44-46d0-4f6a-aaf6-6d690b6138a7. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/17 15:14:06 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

In [21]:
#Stop query when necessary
query.stop()

26/04/17 15:14:56 WARN DAGScheduler: Failed to cancel job group 9aeeb25a-8899-4b5d-8eee-8ae584b3c9fa. Cannot find active jobs for it.
26/04/17 15:14:56 WARN DAGScheduler: Failed to cancel job group 9aeeb25a-8899-4b5d-8eee-8ae584b3c9fa. Cannot find active jobs for it.


We've essentially created a stream that can automatically format the data to our needs!